# 01. Data Understanding

KNHANES 2017-2023 데이터 구조 확인

이 노트북에서는 머신러닝을 하기 전에 원본 데이터가 어떻게 생겼는지 확인한다.

## 목차

- 데이터 파일 확인
- 데이터 불러오기
- 데이터 크기 확인
- 컬럼 이름 확인
- 처음 5행 확인
- 주요 변수 후보 확인

## 1. 라이브러리 불러오기

`pandas`는 표 형태의 데이터를 다룰 때 사용한다.  
KNHANES 원본 파일은 `.sav` 형식이므로 `pyreadstat`을 사용해서 불러온다.

In [ ]:
from pathlib import Path

import pandas as pd
import pyreadstat

## 2. 데이터 파일 확인

먼저 프로젝트 폴더 안에 있는 KNHANES 원본 데이터 파일을 확인한다.

In [ ]:
data_path = Path("분석데이터원본")

file_list = sorted(data_path.glob("*.sav"))

print("데이터 파일 개수:", len(file_list))

for file in file_list:
    print(file.name)

## 3. 데이터 불러오기

데이터프레임(DataFrame)은 행과 열로 구성된 표 형태의 데이터이다.

- 행(row): 한 사람 또는 한 관측치
- 열(column): age, sex, BMI 같은 변수

각 연도별 파일을 하나씩 불러와서 `data_dict`에 저장한다.

In [ ]:
data_dict = {}

for file in file_list:
    print("불러오는 파일:", file.name)
    
    df, meta = pyreadstat.read_sav(file)
    data_dict[file.name] = df
    
    print("데이터 크기:", df.shape)
    print()

## 4. 데이터 크기 확인

`shape`는 데이터의 행과 열 개수를 보여준다.

예: `(8000, 900)`이면 8000행, 900열이라는 뜻이다.

In [ ]:
shape_list = []

for file_name, df in data_dict.items():
    shape_list.append([file_name, df.shape[0], df.shape[1]])

shape_df = pd.DataFrame(shape_list, columns=["file_name", "rows", "columns"])
shape_df

## 5. 컬럼 이름 확인

컬럼 이름을 확인하면 어떤 변수가 데이터에 들어있는지 알 수 있다.  
머신러닝을 하기 전에 사용할 수 있는 변수를 먼저 확인하는 것이 중요하다.

In [ ]:
for file_name, df in data_dict.items():
    print("=" * 80)
    print(file_name)
    print("컬럼 개수:", len(df.columns))
    print(list(df.columns))
    print()

## 6. 처음 5행 확인

`head()`를 사용하면 데이터의 처음 5행을 볼 수 있다.  
값들이 어떤 형태로 들어있는지 간단히 확인할 수 있다.

In [ ]:
for file_name, df in data_dict.items():
    print("=" * 80)
    print(file_name)
    display(df.head())

## 7. 주요 변수 후보 정리

이번 프로젝트는 한국 19-39세 젊은 성인의 고혈압 위험 예측이 목표이다.

그래서 다음과 관련된 변수를 먼저 확인한다.

- 나이, 성별
- BMI
- glucose, triglycerides
- 흡연, 음주, 운동
- 소득, 교육수준
- 혈압

In [ ]:
important_variables = pd.DataFrame({
    "concept": [
        "age",
        "sex",
        "BMI",
        "glucose",
        "triglycerides",
        "smoking",
        "alcohol",
        "exercise",
        "income",
        "household income",
        "education",
        "systolic blood pressure",
        "diastolic blood pressure"
    ],
    "possible_column": [
        "age",
        "sex",
        "HE_BMI",
        "HE_glu",
        "HE_TG",
        "sm_presnt",
        "dr_month",
        "pa_aerobic",
        "incm",
        "ho_incm",
        "educ",
        "HE_sbp",
        "HE_dbp"
    ],
    "simple_meaning": [
        "participant age",
        "male/female",
        "body mass index",
        "fasting glucose",
        "blood triglyceride level",
        "current smoking status",
        "monthly drinking status",
        "aerobic physical activity",
        "individual income level",
        "household income level",
        "education level",
        "systolic blood pressure",
        "diastolic blood pressure"
    ]
})

important_variables

## 8. 주요 변수 존재 여부 확인

연도별 데이터마다 컬럼 구성이 조금 다를 수 있다.  
따라서 주요 변수 후보가 각 파일에 있는지 확인한다.

In [ ]:
check_columns = important_variables["possible_column"].tolist()

check_result = []

for file_name, df in data_dict.items():
    one_result = {"file_name": file_name}
    
    for col in check_columns:
        one_result[col] = col in df.columns
    
    check_result.append(one_result)

check_df = pd.DataFrame(check_result)
check_df

## 9. 주요 변수만 미리 보기

전체 컬럼이 너무 많기 때문에 주요 변수 후보만 선택해서 처음 5행을 확인한다.

In [ ]:
for file_name, df in data_dict.items():
    selected_columns = []
    
    for col in check_columns:
        if col in df.columns:
            selected_columns.append(col)
    
    print("=" * 80)
    print(file_name)
    print("선택된 컬럼:", selected_columns)
    display(df[selected_columns].head())

## 10. 정리

이번 노트북에서는 원본 KNHANES 데이터의 구조를 확인하였다.

확인한 내용:

- 2017-2023년 데이터 파일 목록
- 각 데이터의 행과 열 개수
- 컬럼 이름
- 처음 5행
- 프로젝트에 사용할 가능성이 있는 주요 변수

아직 전처리나 머신러닝은 하지 않았다. 다음 단계에서 데이터를 합치고, 연구 대상인 19-39세만 선택한 후 target 변수를 만들 예정이다.